## Data generation for sparse grid

Generate sparse grid, evaluate QML model on grid points and save data


In [ ]:
# Importing necessary packages (from Lorenzos QML model)
import sys
import os
import importlib
import quasi_interpolation as qi
import pennylane as qml
import pennylane.numpy as np
import jax
from jax import numpy as jnp


jax.config.update("jax_enable_x64", True)

path_base = Path(os.getcwd()) 
# Current path for importing custom functions
sys.path.insert(0, str(path_base / "clc_functions")) 


import qnn_layouts_pennylane
importlib.reload(qnn_layouts_pennylane)
import qnn_layouts_pennylane as pqcs

In [ ]:


# Folder in which to find the test inputs
test_inputs_folder = 'test_data/'


transform_output = True
name_out_transf = ''
if transform_output:
    name_out_transf = '_transformedCLC'

### Interval within which transformed clc should be bounded
bound_output = [0.0, 1.0]

### Upper bound for input transformation
transform_input = True
upperbound = np.pi
upperbound_name = '1p0pi'

batch_size = 100
n_batch_name = str(batch_size)

# Learning rate
learning_rate = 0.001
learning_rate_name = '0p001'

### Kept features
features_kept = ['hus', 'clw', 'cli', 'ta', 'pa', 'hwind']
no_of_features = len(features_kept)

### Architecture specifications
no_qubits = no_of_features


In [ ]:

### PQC architecture layout
# No of shots for circuit evaluation
no_shots = 100 # or #inf
#No of shots that were used in training 
no_shots_training = 'inf'
#Architecture to be used 'ZZXY' or 'XYZ'
name_arch = 'XYZ'
### PQC architecture layout + optimal params
if name_arch == 'XYZ':
    pqc_layout = pqcs.XYZ_circuit;  name_arch = 'XYZ';  n_enc = 4;  n_dec = 2; 
    if no_shots_training == 'inf':
        best_exp = 1 
    else:
        best_exp = 4
elif name_arch == 'ZZXY':
    pqc_layout = pqcs.ZZXY_circuit;  name_arch = 'ZZXY';  n_enc = 2;  n_dec = 5;
    if no_shots_training == 'inf':
        best_exp = 6 
    else:
        best_exp = 3

n_enc_name = str(n_enc)
n_dec_name = str(n_dec)

# Load optimal params:
if no_shots_training == 'inf':
    params_folder = 'optimal_params/'
    namefilepars = 'optimal_params'
    name_end = ('_upperbound' + upperbound_name + name_out_transf + '_' + name_arch + '_Nenc' + n_enc_name + 
                '_Ndec' + n_dec_name + '_batch' + n_batch_name + '_lr' + learning_rate_name + '_test' + str(best_exp))
    filename_pars = namefilepars + name_end + '.npy'
else: #varreg, trainign with 1000t
    params_folder = 'optimal_params/'
    filename_pars = 'optimal_params_Nshots'+str(no_shots_training)+'_multinomial_transformedCLC_' + name_arch + '_Nenc' +str(n_enc) + '_Ndec' + str(n_dec) + '_batch100_alpha0p005_iniparam12345_test' + str(best_exp) +'.npy'
    

path_file = os.path.join(params_folder, filename_pars)
opt_params_np = np.load(path_file)
opt_params = jnp.asarray(opt_params_np)


In [ ]:

### ---------------------------------------------------------------------------------------- ###
## ---------------------------------- Initialize QNN model ---------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

no_gate_angles = pqcs.no_of_angles_pqc(name_arch, no_qubits, n_enc, n_dec)
no_params = no_gate_angles + no_qubits + 1


### Define the quantum device
if no_shots == 'inf':
    dev = qml.device('default.qubit.jax', wires=no_qubits)
else:
    dev = qml.device('default.qubit.jax', wires=no_qubits, shots=no_shots)


### Define pqc with measured observables
@qml.qnode(dev, interface="jax")
def qnn_pqc(inputs, pars):
    pqc_layout(inputs, pars, n_enc=n_enc, n_dec=n_dec, wires=dev.wires)
    return [qml.expval(qml.PauliZ(i)) for i in range(no_qubits)]

### Define the QNN model (pqc + postprocessing)
@jax.jit
def model_qnn(params, inputs):
    # Split paramter vector in the different components 
    no_angles = no_gate_angles
    no_weights = no_qubits
    no_bias = 1
    angles = jax.lax.dynamic_slice(params, [0], [no_angles])
    weights = jax.lax.dynamic_slice(params, [no_angles], [no_weights])
    bias = jax.lax.dynamic_slice(params, [no_angles + no_weights], [no_bias])

    # Computation of the quantum circuit in 'measured_batches'
    # 'measured_batches' contains a no_qubits-long list, 
    # where the i-th element is the jnp.array containing the 
    # Z(i) expectation value over the input batch
    measured_batches = qnn_pqc(inputs, angles)

    # Classical post-processing (weighted avg. + bias)
    weighted_output = weights[0] * measured_batches[0]
    for i in range(1,no_qubits):
        weighted_output = weighted_output + weights[i] * measured_batches[i]
    predictions = weighted_output + bias
    return jnp.squeeze(predictions)

#@jax.jit
def eval_model(data):
    predictions = jnp.array(model_qnn(opt_params,data))
    return predictions



## Load/Generate Sparse grid on [0,1]^d

In [ ]:
L = 8
d = 6
try:
    sparse_grid = np.load('sparse_grid_coo_'+str(L)+'.npy')
    print('Sparse grid loaded')
except:
    print('Generate sparse grid')
    sparse_grid = qi.generate_sparse_grid(L,d)
Y_data = np.empty((sparse_grid.shape[0],))

In [ ]:
#evaluate model on sparse grid

batch_size = 500
no_batches_test = int(np.floor(sparse_grid.shape[0] / batch_size)) ###<----------!!!
for kk in range(0, no_batches_test-1):
    if (kk%400 == 0):
        print(kk)
    ins_batch = sparse_grid[kk*batch_size:(kk+1)*batch_size,:]
    outs_batch= np.array(eval_model(np.pi*2*ins_batch)) # grid scaled to [0,2pi]^d
    Y_data[kk*batch_size:(kk+1)*batch_size] =outs_batch




ins_batch = sparse_grid[(no_batches_test-1)*batch_size:,:]
outs_batch= np.array(eval_model(np.pi*2*ins_batch))

Y_data[(no_batches_test-1)*batch_size:]= outs_batch
print('done') 

np.save('sparse_grid_coo_'+ str(L), sparse_grid) #This is a large file!
np.save('sparse_grid_val_'+ name_arch+ '_'+str(L), Y_data)